<div style="
    font-family: 'Trebuchet MS'; 
    padding: 30px; 
    border-radius: 60px; 
    background: linear-gradient(135deg, rgba(33,87,62,1), rgb(92,152,255));
    color: white;
    text-align: center;
    box-shadow: 0 8px 22px rgba(0,0,0,0.25);
">
    <h1 style="font-family: Trebuchet MS; padding: 12px; font-size: 48px; color:rgba(33, 87, 62, 1); text-align: center; line-height: 1.25;">
    <b>⚽In Match<span style="color: #000000"> Notebook 🎮📉</span></b><br>
  <span style="color: #000000; font-size: 24px">features contained :</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide suggested replacements for the players in the field during match ✨</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide suggested tactical style for the players in the field based on the real-time statistics from the match ✨</span>
</h1>

</div>


In [1]:
api_base = r'https://football-backend-app.victoriouswater-69fff737.swedencentral.azurecontainerapps.io/'

In [17]:
import numpy as np , json , requests
import pandas as pd
from google import genai
from pandas import DataFrame , Series
import ast

In [111]:
import asyncio
import aiohttp


event_id = 13980104

# limit concurrency (important for API stability)
semaphore = asyncio.Semaphore(10)


# ---------- safe async fetch ----------
async def fetch_json(session, url):
    async with semaphore:
        try:
            async with session.get(url) as response:
                # handle non-json responses safely
                if response.content_type != 'application/json':
                    text = await response.text()
                    return None if text.strip() == "" else text

                return await response.json()

        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None


# ---------- fetch player data ----------
async def fetch_player_data(session, player_id):
    stats_url = f'{api_base}/events/{event_id}/player/{player_id}/statistics'
    heatmap_url = f'{api_base}/events/{event_id}/player/{player_id}/heatmap'
    rating_url = f'{api_base}/events/{event_id}/player/{player_id}/rating-breakdown'

    stats, heatmap, rating = await asyncio.gather(
        fetch_json(session, stats_url),
        fetch_json(session, heatmap_url),
        fetch_json(session, rating_url),
        return_exceptions=True
    )

    return player_id , stats, heatmap, rating


# ---------- main async function ----------
async def data_async():
    async with aiohttp.ClientSession() as session:

        # fetch event data in parallel
        event_stats, lineups  , players_shotmaps = await asyncio.gather(
            fetch_json(session, f'{api_base}/events/{event_id}/statistics'),
            fetch_json(session, f'{api_base}/events/{event_id}/lineups') ,
            fetch_json(session , f'{api_base}/events/{event_id}/shotmap')
        )

        players = lineups['home']['players'] + lineups['away']['players']

        # create tasks
        tasks = [
            fetch_player_data(session, player['player']['id'])
            for player in players
        ]

        results = await asyncio.gather(*tasks, return_exceptions=True)

        # containers
        players_stats = []
        players_heatmaps = []
        players_rating_breakdowns = []

        # unpack safely
        for result in results:
            if isinstance(result, Exception):
                print("Task failed:", result)
                continue

            player_id , stats, heatmap, rating = result
            if stats:
                stats['heatmap'] = heatmap.get('heatmap', {}) if heatmap else None
                players_stats.append(stats)
            players_heatmaps.append({
            "player_id": player_id,
            "heatmap": heatmap.get('heatmap', {}) if heatmap else None })
            players_rating_breakdowns.append(rating)

        return (
            event_stats,
            lineups,
            players_stats,
            players_heatmaps,
            players_shotmaps,
            players_rating_breakdowns
        )



In [ ]:
result = await data_async()

event_stats, lineups, players_stats, players_heatmaps, players_shotmaps, players_rating_breakdowns = result

In [ ]:
players_stats


In [ ]:

with open("in_data_examples/event_stats2.json", "w", encoding="utf-8") as f:
    json.dump(event_stats, f, indent=2, ensure_ascii=False)


with open("in_data_examples/lineups2.json", "w", encoding="utf-8") as f:
    json.dump(lineups, f, indent=2, ensure_ascii=False)


with open("in_data_examples/players_stats2.json", "w", encoding="utf-8") as f:
    json.dump(players_stats, f, indent=2, ensure_ascii=False)


with open("in_data_examples/players_shotmaps2.json", "w", encoding="utf-8") as f:
    json.dump(players_shotmaps, f, indent=2, ensure_ascii=False)


with open("in_data_examples/players_heatmaps2.json", "w", encoding="utf-8") as f:
    json.dump(players_heatmaps, f, indent=2, ensure_ascii=False)


with open("in_data_examples/players_rating_breakdowns2.json", "w", encoding="utf-8") as f:
    json.dump(players_rating_breakdowns, f, indent=2, ensure_ascii=False)


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">loading data </div>

In [115]:
import json

# Read event statistics
with open("in_data_examples/event_stats1.json", "r", encoding="utf-8") as f:
    event_stats1 = json.load(f)

# Read team lineups
with open("in_data_examples/lineups1.json", "r", encoding="utf-8") as f:
    lineups1 = json.load(f)

# Read players statistics
with open("in_data_examples/players_stats1.json", "r", encoding="utf-8") as f:
    players_stats1 = json.load(f)

# Read players shotmaps data
with open("in_data_examples/players_shotmaps1.json", "r", encoding="utf-8") as f:
    players_shotmaps1 = json.load(f)

# Read players heatmaps data
with open("in_data_examples/players_heatmaps1.json", "r", encoding="utf-8") as f:
    players_heatmaps1 = json.load(f)

# Read players rating breakdowns
with open("in_data_examples/players_rating_breakdowns1.json", "r", encoding="utf-8") as f:
    players_rating_breakdowns1 = json.load(f)


# Read event statistics
with open("in_data_examples/event_stats2.json", "r", encoding="utf-8") as f:
    event_stats2 = json.load(f)

# Read team lineups
with open("in_data_examples/lineups2.json", "r", encoding="utf-8") as f:
    lineups2 = json.load(f)

# Read players statistics
with open("in_data_examples/players_stats2.json", "r", encoding="utf-8") as f:
    players_stats2 = json.load(f)

# Read players shotmaps data
with open("in_data_examples/players_shotmaps2.json", "r", encoding="utf-8") as f:
    players_shotmaps2 = json.load(f)

# Read players heatmaps data
with open("in_data_examples/players_heatmaps2.json", "r", encoding="utf-8") as f:
    players_heatmaps2 = json.load(f)

# Read players rating breakdowns
with open("in_data_examples/players_rating_breakdowns2.json", "r", encoding="utf-8") as f:
    players_rating_breakdowns2 = json.load(f)



# collecting home data only

In [116]:
def json_to_df(data):
    result = {}
    
    for period_data in data['statistics']:
        period = period_data['period'].lower()  # all, 1st, 2nd
        
        for group in period_data['groups']:
            group_name = group['groupName'].lower().replace(" ", "_")
            
            for item in group['statisticsItems']:
                stat_name = item['name'].lower().replace(" ", "_")
                
                # final column name (no "1")
                col_name = f"{group_name}_{stat_name}_{period}"
                
                # store HOME value
                result[col_name] = item['homeValue']
    
    return pd.DataFrame([result])


# Convert both JSONs
df1 = json_to_df(event_stats1)
df2 = json_to_df(event_stats2)

# Concatenate row-wise
event_stats = pd.concat([df1, df2], axis=0, ignore_index=True).rename(index = {0 : 'snap1' , 1 : 'snap2'})

event_stats.to_csv('in_data_examples/work_data/event_stats.csv' , index=False)

In [117]:
import pandas as pd

def json_players_to_df(data, suffix):
    rows = []
    
    for p in data['players']:
        player_info = p['player']
        stats = p.get('statistics', {})
        
        row = {}
        
        # ✅ Common columns (NO suffix)
        row['id'] = player_info.get('id')
        row['name'] = player_info.get('name')
        row['shirt_number'] = p.get('shirtNumber')
        row['team_id'] = p.get('teamId')
        row['position'] = p.get('position')
        
        # ✅ Stats columns (WITH suffix)
        for key, value in stats.items():
            if isinstance(value, dict):  # skip nested
                continue
            row[f"{key}_{suffix}"] = value
        
        rows.append(row)
    
    return pd.DataFrame(rows)


# ✅ Convert both JSONs
df1 = json_players_to_df(lineups1.get('home'), suffix=1)
df2 = json_players_to_df(lineups2.get('home'), suffix=2)

# ✅ Merge on shared columns
common_cols = ['id', 'name', 'shirt_number', 'team_id', 'position']

lineups_and_stats = pd.merge(df1, df2, on=common_cols, how='outer')
lineups_and_stats.to_csv('in_data_examples/work_data/lineups_and_stats.csv' , index=False)

In [118]:
def shotmap_to_df(data, suffix):
    rows = []
    
    for shot in data['shotmap']:
        # ✅ keep only home team shots
        if not shot.get('isHome'):
            continue
        
        row = {}
        
        # ✅ shared columns (NO suffix)
        row['shot_id'] = shot.get('id')
        row['player_id'] = shot['player'].get('id')
        row['player_name'] = shot['player'].get('name')
        row['time'] = shot.get('time')
        
        # ✅ stats (WITH suffix)
        row[f'shot_type_{suffix}'] = shot.get('shotType')
        row[f'situation_{suffix}'] = shot.get('situation')
        row[f'body_part_{suffix}'] = shot.get('bodyPart')
        row[f'xg_{suffix}'] = shot.get('xg')
        row[f'xgot_{suffix}'] = shot.get('xgot')
        
        # coordinates
        coords = shot.get('playerCoordinates', {})
        row[f'x_{suffix}'] = coords.get('x')
        row[f'y_{suffix}'] = coords.get('y')
        
        rows.append(row)
    
    return pd.DataFrame(rows)


# ✅ convert both jsons
df1 = shotmap_to_df(players_shotmaps1, suffix=1)
df2 = shotmap_to_df(players_shotmaps2, suffix=2)

# ✅ merge on shared columns
common_cols = ['shot_id', 'player_id', 'player_name', 'time']

shotmaps = pd.merge(df1, df2, on=common_cols, how='outer')

shotmaps.to_csv('in_data_examples/work_data/shotmaps.csv' , index=False)

In [119]:
# working on heatmaps to convert them to dataframes
players_heatmaps1 = pd.json_normalize(players_heatmaps1).fillna('[]').rename(columns = {'heatmap' : 'heatmap1'})
players_heatmaps2 = pd.json_normalize(players_heatmaps2).fillna('[]').rename(columns = {'heatmap' : 'heatmap2'})
heatmaps =  pd.merge(players_heatmaps1, players_heatmaps2, on='player_id', how='inner')
heatmaps = heatmaps[heatmaps['player_id'].isin(list(lineups_and_stats['id'])) ]

heatmaps.to_csv('in_data_examples/work_data/heatmaps.csv' , index=False)

In [ ]:
def euclidean_distance(x1, y1, x2, y2):
    if pd.isna(x2) or pd.isna(y2):
        return 0
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)


def get_zone(x):
    if pd.isna(x):
        return None
    if x < 33:
        return 'def'
    elif x < 66:
        return 'mid'
    else:
        return 'att'


# =========================
# ✅ Extract events
# =========================

def extract_events(data, suffix):
    rows = []
    
    for obj in data:
        if obj is None:
            continue
        
        for event_type in ['passes', 'dribbles', 'defensive', 'ball-carries']:
            events = obj.get(event_type, [])
            
            for e in events:
                row = {}
                
                # ✅ shared columns
                row['event_type'] = event_type
                row['action_type'] = e.get('eventActionType')
                row['isHome'] = e.get('isHome')
                
                # coordinates
                start = e.get('playerCoordinates', {})
                end = e.get('passEndCoordinates', {})
                
                x1, y1 = start.get('x'), start.get('y')
                x2, y2 = end.get('x'), end.get('y')
                
                row['x'] = x1
                row['y'] = y1
                row['x_end'] = x2
                row['y_end'] = y2
                
                # ✅ suffix features
                row[f'outcome_{suffix}'] = e.get('outcome')
                row[f'keypass_{suffix}'] = e.get('keypass')
                row[f'isLongBall_{suffix}'] = e.get('isLongBall')
                row[f'isAssist_{suffix}'] = e.get('isAssist')
                
                # ✅ calculated features
                row[f'distance_{suffix}'] = euclidean_distance(x1, y1, x2, y2)
                row[f'progression_{suffix}'] = (x2 - x1) if x2 is not None else 0
                row[f'zone_{suffix}'] = get_zone(x1)
                
                rows.append(row)
    
    return pd.DataFrame(rows)


# =========================
# ✅ Aggregate features
# =========================

def build_features(df, suffix):
    
    features = {}

    # ✅ total counts
    features['total_events'] = len(df)
    
    # ================= PASS FEATURES =================
    passes = df[df['event_type'] == 'passes']
    
    features[f'total_passes_{suffix}'] = len(passes)
    features[f'completed_passes_{suffix}'] = passes[f'outcome_{suffix}'].sum()
    
    features[f'pass_accuracy_{suffix}'] = (
        passes[f'outcome_{suffix}'].sum() / len(passes)
        if len(passes) > 0 else 0
    )
    
    features[f'key_passes_{suffix}'] = passes[f'keypass_{suffix}'].sum()
    features[f'long_balls_{suffix}'] = passes[f'isLongBall_{suffix}'].sum()
    features[f'assists_{suffix}'] = passes[f'isAssist_{suffix}'].sum()
    
    features[f'avg_pass_length_{suffix}'] = passes[f'distance_{suffix}'].mean()
    features[f'avg_progression_{suffix}'] = passes[f'progression_{suffix}'].mean()
    
    # ================= DEFENSIVE =================
    defensive = df[df['event_type'] == 'defensive']
    
    features[f'def_actions_{suffix}'] = len(defensive)
    features[f'interceptions_{suffix}'] = (defensive['action_type'] == 'interception').sum()
    features[f'tackles_{suffix}'] = (defensive['action_type'] == 'tackle').sum()
    features[f'clearances_{suffix}'] = (defensive['action_type'] == 'clearance').sum()
    features[f'blocks_{suffix}'] = (defensive['action_type'] == 'block').sum()
    features[f'recoveries_{suffix}'] = (defensive['action_type'] == 'ball-recovery').sum()
    
    # ================= DRIBBLES =================
    dribbles = df[df['event_type'] == 'dribbles']
    
    features[f'total_dribbles_{suffix}'] = len(dribbles)
    features[f'successful_dribbles_{suffix}'] = dribbles[f'outcome_{suffix}'].sum()
    
    # ================= CARRIES =================
    carries = df[df['event_type'] == 'ball-carries']
    
    features[f'total_carries_{suffix}'] = len(carries)
    features[f'avg_carry_length_{suffix}'] = carries[f'distance_{suffix}'].mean()
    features[f'avg_carry_progression_{suffix}'] = carries[f'progression_{suffix}'].mean()
    
    # ================= SPATIAL =================
    features[f'final_third_actions_{suffix}'] = (df[f'zone_{suffix}'] == 'att').sum()
    
    return pd.DataFrame([features])


# =========================
# ✅ MAIN PIPELINE
# =========================

def process_two_files(data1, data2):
    
    # extract event-level data
    df1 = extract_events(data1, suffix=1)
    df2 = extract_events(data2, suffix=2)
    
    # aggregate into features
    f1 = build_features(df1, suffix=1)
    f2 = build_features(df2, suffix=2)
    
    # combine horizontally
    final_df = pd.concat([f1, f2], axis=1)
    
    # remove duplicated columns
    final_df = final_df.loc[:, ~final_df.columns.duplicated()]
    
    return df1, df2, final_df


# run

events_df1, events_df2, features_df = process_two_files(players_rating_breakdowns1, players_rating_breakdowns2 )

features_df.to_csv('in_data_examples/work_data/features_from_rates.csv' , index=False)
events_df1

,event_type,action_type,isHome,x,y,x_end,y_end,outcome_1,keypass_1,isLongBall_1,isAssist_1,distance_1,progression_1,zone_1
0,passes,pass,True,7.3,49.8,8.4,27.4,True,False,None,None,22.426993,1.1,def
1,passes,pass,True,5.2,49.8,12.2,10.2,True,False,None,None,40.213928,7.0,def
2,passes,pass,True,22.9,44.5,26.5,77.1,True,False,None,None,32.798171,3.6,def
3,passes,pass,True,7.6,47.8,22.9,95.6,True,False,True,None,50.188943,15.3,def
4,passes,pass,True,13.3,50.3,64.8,75.1,True,False,True,None,57.160213,51.5,def
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,ball-carries,ball-carry,False,53.5,22.0,57.0,35.5,None,None,None,None,13.946326,3.5,mid
1254,ball-carries,ball-carry,False,28.3,21.3,14.6,56.0,None,None,None,None,37.306568,-13.7,def
1255,passes,pass,False,53.0,69.7,46.1,74.4,True,False,None,None,8.348653,-6.9,mid
1256,passes,pass,False,33.3,64.1,37.4,60.6,True,False,None,None,5.390733,4.1,mid


In [121]:
events_df2


,event_type,action_type,isHome,x,y,x_end,y_end,outcome_2,keypass_2,isLongBall_2,isAssist_2,distance_2,progression_2,zone_2
0,passes,pass,True,7.3,49.8,8.4,27.4,True,False,None,None,22.426993,1.1,def
1,passes,pass,True,5.2,49.8,12.2,10.2,True,False,None,None,40.213928,7.0,def
2,passes,pass,True,22.9,44.5,26.5,77.1,True,False,None,None,32.798171,3.6,def
3,passes,pass,True,7.6,47.8,22.9,95.6,True,False,True,None,50.188943,15.3,def
4,passes,pass,True,13.3,50.3,64.8,75.1,True,False,True,None,57.160213,51.5,def
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458,ball-carries,ball-carry,False,50.4,87.4,53.4,87.3,None,None,None,None,3.001666,3.0,mid
1459,passes,cross,False,90.7,13.5,94.9,58.9,None,False,None,None,45.593859,4.2,att
1460,passes,pass,False,46.1,37.4,53.5,49.5,True,True,None,None,14.183441,7.4,mid
1461,defensive,tackle,False,18.0,13.9,NaN,NaN,None,False,None,None,0.000000,0.0,def


In [122]:
features_df

,total_events,total_passes_1,completed_passes_1,pass_accuracy_1,key_passes_1,long_balls_1,assists_1,avg_pass_length_1,avg_progression_1,def_actions_1,...,tackles_2,clearances_2,blocks_2,recoveries_2,total_dribbles_2,successful_dribbles_2,total_carries_2,avg_carry_length_2,avg_carry_progression_2,final_third_actions_2
0,1258,857,733,0.855309,16,81,3,21.739725,4.432555,149,...,28,45,10,82,19,5,289,9.900532,4.010727,288


# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">1. | Rolling Snapshot & Delta Engine </div>

In [ ]:
def players_level_delta(players_df : DataFrame)-> DataFrame :
    """
    For each player, compute delta = snapshot_2 − snapshot_1 for every
    numeric stat column.

    Parameters
    ----------
    players_df : DataFrame
        Must contain columns ending in ``_1`` and ``_2`` (two snapshots)
        plus shared columns: id, name, shirt_number, team_id, position.

    Returns
    -------
    DataFrame  with columns:
        - shared identifiers (id, name, position …)
        - every stat as ``<stat>_delta``
    """
    # shared_columns that won't be computed
    shared_cols = ["id", "name", "shirt_number", "team_id", "position"]

    # Identify all stat bases that have both _1 and _2 variants
    cols_1 = {c.rsplit("_", 1)[0] for c in players_df.columns if c.endswith("_1")}
    cols_2 = {c.rsplit("_", 1)[0] for c in players_df.columns if c.endswith("_2")}
    stat_bases = sorted(cols_1 & cols_2)

    # Build all columns in a dict first to avoid DataFrame fragmentation
    col_data = {}
    for base in stat_bases:
        c1, c2 = f"{base}_1", f"{base}_2"
        col_data[f"{base}_delta"] = (
            pd.to_numeric(players_df[c2], errors="coerce").values
            - pd.to_numeric(players_df[c1], errors="coerce").values
        )

    result = pd.concat(
        [players_df[shared_cols].reset_index(drop=True),
         pd.DataFrame(col_data)],
        axis=1,
    )

    return result 

players = pd.read_csv('in_data_examples\work_data\lineups_and_stats.csv')
output = players_level_delta(players)
output

,id,name,shirt_number,team_id,position,accurateCross_delta,accurateLongBalls_delta,accurateOppositionHalfPasses_delta,accurateOwnHalfPasses_delta,accuratePass_delta,...,totalPass_delta,totalProgression_delta,totalProgressiveBallCarriesDistance_delta,totalShots_delta,totalTackle_delta,touches_delta,unsuccessfulTouch_delta,wasFouled_delta,wonContest_delta,wonTackle_delta
0,259281,Alberto Paleari,1,2696,G,NaN,1.0,0.0,2.0,2.0,...,3.0,1.991279,NaN,0,NaN,4.0,NaN,NaN,NaN,NaN
1,1089453,Saúl Coco,23,2696,D,NaN,0.0,0.0,0.0,0.0,...,0.0,-0.000022,-0.000018,0,0.0,0.0,NaN,NaN,NaN,0.0
2,840414,Ardian Ismajli,44,2696,D,NaN,0.0,2.0,3.0,5.0,...,5.0,4.308123,2.332135,0,0.0,7.0,NaN,0.0,NaN,0.0
3,848279,Enzo Ebosse,77,2696,D,NaN,0.0,2.0,3.0,5.0,...,5.0,2.086005,NaN,0,0.0,6.0,0.0,NaN,NaN,NaN
4,233826,Valentino Lazaro,20,2696,M,NaN,0.0,0.0,0.0,0.0,...,0.0,0.000033,0.000041,0,0.0,0.0,0.0,NaN,0.0,NaN
5,1142612,Emirhan İlkhan,6,2696,M,0.0,1.0,2.0,1.0,3.0,...,4.0,1.411271,NaN,0,NaN,5.0,NaN,0.0,NaN,NaN
6,1030829,Gvidas Gineitis,66,2696,M,NaN,0.0,0.0,0.0,0.0,...,0.0,0.000026,NaN,0,0.0,0.0,0.0,0.0,NaN,0.0
7,1142692,Rafael Obrador,33,2696,M,0.0,0.0,1.0,1.0,2.0,...,3.0,4.203261,1.660871,0,0.0,5.0,NaN,NaN,NaN,0.0
8,357596,Nikola Vlašić,10,2696,F,NaN,NaN,2.0,1.0,4.0,...,4.0,4.778476,3.618803,0,0.0,5.0,0.0,0.0,0.0,NaN
9,773409,Che Adams,19,2696,F,NaN,0.0,0.0,0.0,0.0,...,0.0,0.000021,-0.000016,0,0.0,0.0,NaN,NaN,NaN,0.0


In [ ]:
def team_level_delta(team_df : DataFrame ) -> DataFrame :
    """
    Compute delta between the two snapshot rows of team-level event
    statistics (row 0 = snap1, row 1 = snap2).

    Returns a single-row DataFrame with ``<col>_delta`` columns.
    """
    # checking for the existense of the two snapshots
    if len(team_df) < 2:
        raise ValueError("event_stats must have at least 2 rows (snap1, snap2)")

    # getting the two snapshots separated
    snap1 = pd.to_numeric(team_df.iloc[0], errors="coerce")
    snap2 = pd.to_numeric(team_df.iloc[1], errors="coerce")
    delta = snap2 - snap1

    # constructing the final dataframe
    delta_df = pd.DataFrame([delta.values], columns=[f"{c}_delta" for c in team_df.columns])
 

    return delta_df

events = pd.read_csv('in_data_examples\work_data\event_stats.csv')
output = team_level_delta(events)
output

,match_overview_ball_possession_all_delta,match_overview_expected_goals_all_delta,match_overview_big_chances_all_delta,match_overview_total_shots_all_delta,match_overview_goalkeeper_saves_all_delta,match_overview_corner_kicks_all_delta,match_overview_fouls_all_delta,match_overview_passes_all_delta,match_overview_tackles_all_delta,match_overview_free_kicks_all_delta,...,duels_ground_duels_2nd_delta,duels_aerial_duels_2nd_delta,duels_dribbles_2nd_delta,defending_tackles_won_2nd_delta,defending_total_tackles_2nd_delta,defending_interceptions_2nd_delta,defending_recoveries_2nd_delta,defending_clearances_2nd_delta,goalkeeping_total_saves_2nd_delta,goalkeeping_goal_kicks_2nd_delta
0,4.0,0.27,0.0,2.0,0.0,1.0,1.0,38.0,2.0,1.0,...,4.0,1.0,1.0,1.0,2.0,1.0,5.0,3.0,0.0,0.0


In [18]:
def _parse_heatmap(raw) -> list[dict]:
    """Safely parse a heatmap column value into a list of {x, y} dicts."""
    if isinstance(raw, list):
        return raw
    if isinstance(raw, str):
        try:
            parsed = ast.literal_eval(raw)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []
    
def heatmap_delta(heatmaps: pd.DataFrame) -> pd.DataFrame:
    """
    For each player, compute the *new* heatmap points that appeared
    between snapshot 1 and snapshot 2.

    Strategy: if snap1 had N points and snap2 has M (M ≥ N), take the
    last (M − N) points from snap2 as the delta.

    Returns DataFrame with columns: player_id, heatmap_delta (list),
    n_snap1, n_snap2, n_new_points, centroid_snap1_x/y, centroid_snap2_y/y,
    centroid_delta_x/y.
    """
    rows = []
    for _, row in heatmaps.iterrows():
        pid = row["player_id"]
        h1 = _parse_heatmap(row.get("heatmap1", "[]"))
        h2 = _parse_heatmap(row.get("heatmap2", "[]"))

        n1, n2 = len(h1), len(h2)
        n_new = max(0, n2 - n1)
        delta_points = h2[-n_new:] if n_new > 0 else []

        # Centroids
        def centroid(pts):
            if not pts:
                return np.nan, np.nan
            xs = [p["x"] for p in pts]
            ys = [p["y"] for p in pts]
            return np.mean(xs), np.mean(ys)

        cx1, cy1 = centroid(h1)
        cx2, cy2 = centroid(h2)

        rows.append({
            "player_id": pid,
            "heatmap_delta": delta_points,
            "n_snap1": n1,
            "n_snap2": n2,
            "n_new_points": n_new,
            "centroid_snap1_x": cx1,
            "centroid_snap1_y": cy1,
            "centroid_snap2_x": cx2,
            "centroid_snap2_y": cy2,
            "centroid_delta_x": cx2 - cx1 if not (np.isnan(cx1) or np.isnan(cx2)) else 0,
            "centroid_delta_y": cy2 - cy1 if not (np.isnan(cy1) or np.isnan(cy2)) else 0,
        })

    return pd.DataFrame(rows)


heatmaps = pd.read_csv(r'in_data_examples\work_data\heatmaps.csv')
output = heatmap_delta(heatmaps)
output

,player_id,heatmap_delta,n_snap1,n_snap2,n_new_points,centroid_snap1_x,centroid_snap1_y,centroid_snap2_x,centroid_snap2_y,centroid_delta_x,centroid_delta_y
0,259281,"[{'x': 5, 'y': 56}, {'x': 5, 'y': 52}, {'x': 1...",47,53,6,10.021277,50.319149,10.075472,50.830189,0.054195,0.511040
1,1089453,[],42,42,0,32.833333,22.285714,32.833333,22.285714,0.000000,0.000000
2,840414,"[{'x': 32, 'y': 29}, {'x': 48, 'y': 48}, {'x':...",60,68,8,33.950000,40.516667,33.294118,40.720588,-0.655882,0.203922
3,848279,"[{'x': 23, 'y': 68}, {'x': 26, 'y': 83}, {'x':...",61,69,8,36.409836,71.901639,36.420290,72.086957,0.010454,0.185317
4,233826,[],26,26,0,58.423077,13.884615,58.423077,13.884615,0.000000,0.000000
5,1142612,"[{'x': 29, 'y': 68}, {'x': 68, 'y': 38}, {'x':...",40,45,5,53.825000,44.300000,51.311111,45.733333,-2.513889,1.433333
6,1030829,[],38,38,0,55.973684,66.315789,55.973684,66.315789,0.000000,0.000000
7,1142692,"[{'x': 26, 'y': 88}, {'x': 68, 'y': 62}, {'x':...",40,46,6,55.200000,79.175000,53.760870,77.804348,-1.439130,-1.370652
8,357596,"[{'x': 53, 'y': 88}, {'x': 88, 'y': 50}, {'x':...",51,58,7,56.960784,33.039216,57.068966,37.258621,0.108181,4.219405
9,773409,[],31,31,0,48.516129,73.612903,48.516129,73.612903,0.000000,0.000000


In [19]:
def shotmaps_delta(shotmaps_df : DataFrame) -> DataFrame:
    """
    Identify shots that exist only in snapshot 2 (new shots in the window).

    Strategy: shots present in both snapshots have matching ``_1`` and ``_2``
    columns. Shots that are *new* to snapshot 2 will have NaN in ``_1``
    columns but values in ``_2``.

    Also returns all shots with a computed ``is_new`` flag.
    """
    df = shotmaps_df.copy()

    # A shot is new to snapshot 2 if it has _2 data but no _1 data
    has_1 = df["xg_1"].notna()
    has_2 = df["xg_2"].notna()

    df["is_new_in_snap2"] = (~has_1) & has_2
    df["is_in_both"] = has_1 & has_2

    return df

shotmaps = pd.read_csv(r'in_data_examples\work_data\shotmaps.csv')
output = shotmaps_delta(shotmaps)
output

,shot_id,player_id,player_name,time,shot_type_1,situation_1,body_part_1,xg_1,xgot_1,x_1,y_1,shot_type_2,situation_2,body_part_2,xg_2,xgot_2,x_2,y_2,is_new_in_snap2,is_in_both
0,7148388,357596,Nikola Vlašić,79,goal,penalty,right-foot,0.788400,0.987646,11.5,50.0,goal,penalty,right-foot,0.788400,0.987646,11.5,50.0,False,True
1,7148384,38162,Duván Zapata,75,block,assisted,head,0.038838,0.000000,10.2,56.0,block,assisted,head,0.038838,0.000000,10.2,56.0,False,True
2,7148369,38162,Duván Zapata,74,block,corner,right-foot,0.095032,0.000000,8.6,48.7,block,corner,right-foot,0.095032,0.000000,8.6,48.7,False,True
3,7148362,341143,Giovanni Simeone,74,miss,corner,head,0.142324,0.000000,8.3,51.1,miss,corner,head,0.142324,0.000000,8.3,51.1,False,True
4,7148346,341143,Giovanni Simeone,72,miss,fast-break,right-foot,0.328793,0.000000,9.9,52.8,miss,fast-break,right-foot,0.328793,0.000000,9.9,52.8,False,True
5,7148335,341143,Giovanni Simeone,70,goal,fast-break,right-foot,0.691169,0.360194,12.0,52.8,goal,fast-break,right-foot,0.691169,0.360194,12.0,52.8,False,True
6,7148255,1030829,Gvidas Gineitis,56,block,free-kick,left-foot,0.034802,0.000000,28.5,56.7,block,free-kick,left-foot,0.034802,0.000000,28.5,56.7,False,True
7,7147938,1142692,Rafael Obrador,38,miss,regular,left-foot,0.025780,0.000000,12.8,22.1,miss,regular,left-foot,0.025780,0.000000,12.8,22.1,False,True
8,7147936,1030829,Gvidas Gineitis,38,block,regular,head,0.020264,0.000000,13.6,49.3,block,regular,head,0.020264,0.000000,13.6,49.3,False,True
9,7147664,1089453,Saúl Coco,8,miss,assisted,right-foot,0.099231,0.000000,14.8,52.1,miss,assisted,right-foot,0.099231,0.000000,14.8,52.1,False,True


In [16]:
def features_delta(features_df : DataFrame ) -> DataFrame:
    """
    Compute deltas for aggregated event features (passes, dribbles,
    defensive actions, carries).

    The input has columns like ``total_passes_1`` and ``total_passes_2``.
    Output adds ``total_passes_delta`` = _2 − _1 for each.
    """
    cols_1 = {c.rsplit("_", 1)[0] for c in features_df.columns
              if c.endswith("_1") and c != "total_events"}
    cols_2 = {c.rsplit("_", 1)[0] for c in features_df.columns
              if c.endswith("_2")}
    bases = sorted(cols_1 & cols_2)

    data = features_df.copy()
    result = pd.DataFrame()
    result['total_events'] = data['total_events']
    for base in bases:
        c1, c2 = f"{base}_1", f"{base}_2"
        result[f"{base}_delta"] = (
            pd.to_numeric(data[c2], errors="coerce")
            - pd.to_numeric(data[c1], errors="coerce")
        )
    return result

features = pd.read_csv(r'in_data_examples\work_data\features_from_rates.csv')
output = features_delta(features)
output

,total_events,assists_delta,avg_carry_length_delta,avg_carry_progression_delta,avg_pass_length_delta,avg_progression_delta,blocks_delta,clearances_delta,completed_passes_delta,def_actions_delta,...,interceptions_delta,key_passes_delta,long_balls_delta,pass_accuracy_delta,recoveries_delta,successful_dribbles_delta,tackles_delta,total_carries_delta,total_dribbles_delta,total_passes_delta
0,1258,0,0.096022,0.217393,0.48165,0.275658,1,7,89,32,...,3,4,23,-0.011367,16,1,5,49,7,117


In [24]:
def all_deltas(players_df, team_df, heatmaps_df, shotmaps_df, features_df):
    '''Accepts dataframes and returns delta dataframes'''

    try:
     
        players_delta = players_level_delta(players_df) if players_df is not None and not players_df.empty else None
        team_delta = team_level_delta(team_df) if team_df is not None and not team_df.empty else None
        heatmaps_delta = heatmap_delta(heatmaps_df) if heatmaps_df is not None and not heatmaps_df.empty else None
        

        shotmaps_delta_result = shotmaps_delta(shotmaps_df) if shotmaps_df is not None and not shotmaps_df.empty else None
        features_delta_result = features_delta(features_df) if features_df is not None and not features_df.empty else None

        return players_delta, team_delta, heatmaps_delta, shotmaps_delta_result, features_delta_result

    except Exception as e:
        print("Error inside all_deltas:", e)
        return None, None, None, None, None

players = pd.read_csv('in_data_examples\work_data\lineups_and_stats.csv')
events = pd.read_csv('in_data_examples\work_data\event_stats.csv')
heatmaps = pd.read_csv(r'in_data_examples\work_data\heatmaps.csv')
shotmaps = pd.read_csv(r'in_data_examples\work_data\shotmaps.csv')
features = pd.read_csv(r'in_data_examples\work_data\features_from_rates.csv')

player_delta , events_delta , heatmaps_delta , shotmaps_delta , features_delta = all_deltas(players , events , heatmaps , shotmaps , features)

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">2. | Feature Engineering & Baseline Comparisons  </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">3. | Performance Deviation (Z-Score Model)   </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">4. | Substitution Urgency XGBoost   </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">5. | Pitch Grid Zone Threat Regressor    </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">6. | Formation Effectiveness Model     </div>

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">7. | Inference Pipeline & LLM Aggregation    </div>